# BABEL

In [40]:
BABEL_PATH = r"/ssd/sqlite/BABEL.db"

level: str = "L1"
text_input = "galactose"

from sqlite_utils import Database
from collections import Counter
from typing import Optional
import time

db = Database(BABEL_PATH)
db.enable_wal()

prioritize: frozenset[str] = frozenset(["SmallMolecule"])
prioritize_placeholders: Optional[str] = (", ".join([f":prioritize{idx}" for idx in range(len(prioritize))]) if prioritize else None)

avoid: frozenset[str] = frozenset(["OrganismTaxon", "BiologicalProcess"])
avoid_placeholders: Optional[str] = (", ".join([f":avoid{idx}" for idx in range(len(avoid))]) if avoid else None)

taxon: str = str(9606)

CategoryFrequency: Counter[str] = Counter()
CategoryFrequency["disease"] += 1

top_logged_category: list[str] = CategoryFrequency.most_common(1)
most_common: Optional[str] = str(top_logged_category[0][0]) if top_logged_category else None

sql_params: dict[str, str] = {"input": text_input.lower()}
if prioritize: sql_params.update({f"prioritize{idx}": category for idx, category in enumerate(prioritize)})
if avoid: sql_params.update({f"avoid{idx}": category for idx, category in enumerate(avoid)})
if taxon: sql_params["taxon"] = taxon
if most_common: sql_params["most_common"] = most_common

sql: str = f"""
    SELECT
        NAMES.CURIE,
        NAMES.CATEGORY,
        NAMES.NAME,
        NAMES.TAXON
    FROM SYNONYMS
    INNER JOIN NAMES ON SYNONYMS.CURIE = NAMES.CURIE
    WHERE 
        {"SYNONYMS.L1 = :input" if level == "L1" else "SYNONYMS.L2 = :input" if level == "L2" else "SYNONYMS.L3 = :input"}
        {"AND (NAMES.CATEGORY != 'Gene' OR NAMES.TAXON = :taxon)" if taxon else ""}
        {f"AND NAMES.CATEGORY NOT IN ({avoid_placeholders})" if avoid_placeholders else ""}
    {f"ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) AND NAMES.CATEGORY = :most_common THEN 0 \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) THEN 1 \n\t\t WHEN NAMES.CATEGORY = :most_common THEN 2 \n\t\t ELSE 3 \n\t END" if prioritize_placeholders and most_common else f"ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) THEN 0 \\n\t\t ELSE 1 \n\t END" if prioritize_placeholders else "ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY = :most_common THEN 0 \n\t\t ELSE 1 \n\t END" if most_common else ""}
    """

start = time.time()
rows = list(db.query(sql, sql_params))
print(time.time() - start)
print(sql)
print(sql_params)
print(rows)


0.0008983612060546875

    SELECT
        NAMES.CURIE,
        NAMES.CATEGORY,
        NAMES.NAME,
        NAMES.TAXON
    FROM SYNONYMS
    INNER JOIN NAMES ON SYNONYMS.CURIE = NAMES.CURIE
    WHERE 
        SYNONYMS.L1 = :input
        AND (NAMES.CATEGORY != 'Gene' OR NAMES.TAXON = :taxon)
        AND NAMES.CATEGORY NOT IN (:avoid0, :avoid1)
    ORDER BY 
	 CASE 
		 WHEN NAMES.CATEGORY IN (:prioritize0) AND NAMES.CATEGORY = :most_common THEN 0 
		 WHEN NAMES.CATEGORY IN (:prioritize0) THEN 1 
		 WHEN NAMES.CATEGORY = :most_common THEN 2 
		 ELSE 3 
	 END
    
{'input': 'galactose', 'prioritize0': 'SmallMolecule', 'avoid0': 'BiologicalProcess', 'avoid1': 'OrganismTaxon', 'taxon': '9606', 'most_common': 'disease'}
[{'CURIE': 'CHEBI:42905', 'CATEGORY': 'SmallMolecule', 'NAME': 'L-Galactose', 'TAXON': ''}, {'CURIE': 'CHEBI:28061', 'CATEGORY': 'SmallMolecule', 'NAME': 'galactose', 'TAXON': ''}, {'CURIE': 'CHEBI:17118', 'CATEGORY': 'SmallMolecule', 'NAME': 'Galactose', 'TAXON': ''}, {'CURI

# KG2

In [ ]:
KG2_PATH = r"/ssd/sqlite/KG2.10.1.db"

db2 = Database(KG2_PATH)
db2.enable_wal()

sql: str = f"""
    SELECT
        clusters.cluster_id,
        clusters.category,
        clusters.name
    FROM nodes
    INNER JOIN clusters ON nodes.cluster_id = clusters.cluster_id
    WHERE
        {"nodes.name = :input" if level == "L1" else "nodes.name_simplified = :input"}
        {f"AND clusters.category NOT IN ({avoid_placeholders})" if avoid_placeholders else ""}
    {f"ORDER BY \n\t CASE \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) AND clusters.category = :most_common THEN 0 \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) THEN 1 \n\t\t WHEN clusters.category = :most_common THEN 2 \n\t\t ELSE 3 \n\t END" if prioritize_placeholders and most_common else f"ORDER BY \n\t CASE \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) THEN 0 \\n\t\t ELSE 1 \n\t END" if prioritize_placeholders else "ORDER BY \n\t CASE \n\t\t WHEN clusters.category = :most_common THEN 0 \n\t\t ELSE 1 \n\t END" if most_common else ""}
    """

sql_params["input"] = text_input

start = time.time()
rows = list(db2.query(sql, sql_params))
print(time.time() - start)
print(sql)
print(sql_params)
print(rows)

0.003074169158935547

    SELECT
        clusters.cluster_id,
        clusters.category,
        clusters.name
    FROM nodes
    INNER JOIN clusters ON nodes.cluster_id = clusters.cluster_id
    WHERE
        nodes.name = :input
        AND clusters.category NOT IN (:avoid0, :avoid1)
    ORDER BY 
	 CASE 
		 WHEN clusters.category IN (:prioritize0) AND clusters.category = :most_common THEN 0 
		 WHEN clusters.category IN (:prioritize0) THEN 1 
		 WHEN clusters.category = :most_common THEN 2 
		 ELSE 3 
	 END
    
{'input': 'galactose', 'prioritize0': 'SmallMolecule', 'avoid0': 'BiologicalProcess', 'avoid1': 'OrganismTaxon', 'most_common': 'disease'}
[{'cluster_id': 'CHEBI:28061', 'category': 'SmallMolecule', 'name': 'alpha-D-galactose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 'name': 'D-galactopyranose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 'name': 'D-galactopyranose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 'name': 'D-galact